# 📘 The AI Engineer's LLM Workbook

**14 Chapters · 14 Google Colab Notebooks · Beginner to Production**

---

*© 2026 JAWNVION LLC — www.jawnvion.com — peter@jawnvion.com*

*Licensed for individual use. Do not redistribute.*

---

## What's Inside

| # | Chapter |
|---|---------|
| 01 | AI Fundamentals & Problem Framing |
| 02 | Data Science Toolkit (NumPy, Pandas, Matplotlib) |
| 03 | Neural Networks from Scratch |
| 04 | Transformers Architecture Deep Dive |
| 05 | HuggingFace & Pre-Trained Models |
| 06 | QLoRA Fine-Tuning |
| 07 | DPO Alignment Training |
| 08 | Retrieval-Augmented Generation (RAG) |
| 09 | Model Evaluation & Benchmarking |
| 10 | FastAPI Deployment |
| 11 | Monitoring & Observability |
| 12 | Security for AI Systems |
| 13 | Cost Optimization & Quantization |
| 14 | Capstone: End-to-End LLM Project |

---

> **How to use:** Click **Runtime → Run All** in Google Colab, or run cells one at a time.
> Each chapter builds on the last — complete them in order for best results.

---


# Chapter 13: Cost Optimisation & Model Compression
**JAWNVION LLC — AI Training Workbook**

Every token you generate costs money — GPU time. This chapter teaches you to
reduce that cost without sacrificing accuracy, using **quantization**: representing
model weights at lower numeric precision. A 4-bit model is 8× smaller than fp32
and runs 2–4× faster on the same hardware.

**What you'll learn:**
- Why quantization works (information theory perspective)
- Benchmark fp16 vs INT8 vs INT4: VRAM, tokens/sec, and output quality
- Dynamic quantization for CPU/edge deployment (no GPU required)
- Model size on disk at each precision
- Cost calculator: GPU $/hr → cost per 1,000 tokens served
- Precision selection guide: which format for which deployment scenario
- Real cloud GPU pricing for T4, A10G, A100, and GovCloud equivalents

In [ ]:
# — Cell 1: GPU Check ——————————————————————————————————
import torch, subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✓  GPU detected:', result.stdout.strip())
    gpu_name = result.stdout.strip().split(',')[0].strip()
else:
    print('⚠  No GPU — INT8/INT4 benchmarks will run on CPU (much slower)')
    print('   Runtime → Change runtime type → T4 GPU for accurate benchmarks')
    gpu_name = "CPU"

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'   PyTorch : {torch.__version__}')
print(f'   Device  : {device}')
print(f'   CUDA    : {torch.version.cuda if torch.cuda.is_available() else "N/A"}')

In [ ]:
# — Cell 2: Install Packages ——————————————————————————
!pip install -q "tokenizers>=0.22,<0.24"
!pip install -q -U transformers accelerate bitsandbytes
print('✓  Packages installed')
print('   bitsandbytes provides GPU-accelerated INT8 and INT4 quantization')

In [ ]:
# — Cell 3: Imports & Configuration ———————————————————
import torch
import time
import gc
import os
import statistics
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

BASE_MODEL  = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MERGED_DIR  = "/content/tinyllama-merged"   # Chapter 10 merged checkpoint (preferred)
N_BENCH     = 3                             # inference runs per configuration
MAX_NEW     = 80                            # tokens generated per run
BENCH_PROMPT = (
    "### Instruction:\n"
    "Explain the difference between a compiler and an interpreter "
    "in two sentences.\n\n"
    "### Response:\n"
)

print('✓  Config ready')
print(f'   Base model   : {BASE_MODEL}')
print(f'   Bench prompt : {BENCH_PROMPT[:70].strip()}...')
print(f'   Runs / config: {N_BENCH}  |  Max new tokens: {MAX_NEW}')

In [ ]:
# — Cell 4: Benchmark Helper Functions ————————————————

def vram_used_gb() -> float:
    """Current allocated VRAM in GB (0 if no GPU)."""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1e9
    return 0.0

def clear_model(mdl=None):
    """Release model from GPU memory before loading the next one."""
    if mdl is not None:
        del mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def benchmark(model, tokenizer, prompt: str, n: int, max_new: int) -> dict:
    """
    Run n inference passes. Returns dict with latency stats and tokens/sec.
    """
    latencies = []
    tps_list  = []
    last_text = ""
    for _ in range(n):
        inputs = tokenizer(
            prompt, return_tensors="pt", truncation=True, max_length=512
        ).to(model.device)
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new,
                do_sample=False,           # greedy — deterministic for fair comparison
                repetition_penalty=1.3,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
            )
        elapsed = time.time() - t0
        n_new = out.shape[1] - inputs["input_ids"].shape[1]
        latencies.append(elapsed)
        tps_list.append(n_new / elapsed)
        last_text = tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip()
    return {
        "latency_mean":   round(statistics.mean(latencies), 2),
        "latency_median": round(statistics.median(latencies), 2),
        "tps_mean":       round(statistics.mean(tps_list), 1),
        "last_response":  last_text,
    }

def model_size_bytes(model) -> int:
    """Total bytes occupied by model parameters."""
    return sum(p.numel() * p.element_size() for p in model.parameters())

print('✓  Helper functions ready: benchmark(), vram_used_gb(), clear_model()')

In [ ]:
# — Cell 5: Baseline — fp16 (Half Precision) ——————————
# fp16 is the standard production precision for GPU inference.
# Each weight is 2 bytes (vs 4 for fp32). TinyLlama 1.1B ≈ 2.2 GB VRAM.

results = {}   # accumulate all configurations here

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

model_path = MERGED_DIR if os.path.isdir(MERGED_DIR) else BASE_MODEL
print(f"Loading fp16 model from {model_path}...")

torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None
vram_before = vram_used_gb()

model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
)
model_fp16.eval()

vram_after = vram_used_gb()
param_gb   = model_size_bytes(model_fp16) / 1e9

print(f"\nBenchmarking fp16  ({N_BENCH} runs)...")
stats = benchmark(model_fp16, tokenizer, BENCH_PROMPT, N_BENCH, MAX_NEW)

results["fp16"] = {
    "precision":     "fp16  (2 bytes/weight)",
    "vram_gb":       round(vram_after - vram_before, 2),
    "param_gb":      round(param_gb, 2),
    **stats,
}

print(f"  VRAM delta   : {results['fp16']['vram_gb']:.2f} GB")
print(f"  Param size   : {results['fp16']['param_gb']:.2f} GB")
print(f"  Latency mean : {stats['latency_mean']:.2f}s")
print(f"  Throughput   : {stats['tps_mean']:.1f} tok/s")
print(f"  Sample output: {stats['last_response'][:100]}...")

clear_model(model_fp16)
model_fp16 = None
print("\n✓  fp16 baseline complete — model released from VRAM")

In [ ]:
# — Cell 6: INT8 Quantization ————————————————————————
# INT8 = 1 byte per weight. BitsAndBytesConfig handles the quantization
# transparently — the model API is identical to fp16.
# Activations remain in fp16; only weights are compressed.
# Expected: ~1.1 GB VRAM, 5–15% slower than fp16 (dequant overhead).

print("Loading INT8 model...")
vram_before = vram_used_gb()

bnb_int8 = BitsAndBytesConfig(load_in_8bit=True)
model_int8 = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_int8,
    device_map="auto",
)
model_int8.eval()

vram_after = vram_used_gb()
# INT8 weights are stored as int8 but counted as 1 byte — compute manually
param_gb = sum(
    p.numel() * (1 if p.dtype == torch.int8 else p.element_size())
    for p in model_int8.parameters()
) / 1e9

print(f"Benchmarking INT8  ({N_BENCH} runs)...")
stats = benchmark(model_int8, tokenizer, BENCH_PROMPT, N_BENCH, MAX_NEW)

results["int8"] = {
    "precision":     "INT8  (1 byte/weight)",
    "vram_gb":       round(vram_after - vram_before, 2),
    "param_gb":      round(param_gb, 2),
    **stats,
}

print(f"  VRAM delta   : {results['int8']['vram_gb']:.2f} GB")
print(f"  Param size   : {results['int8']['param_gb']:.2f} GB")
print(f"  Latency mean : {stats['latency_mean']:.2f}s")
print(f"  Throughput   : {stats['tps_mean']:.1f} tok/s")
print(f"  Sample output: {stats['last_response'][:100]}...")

clear_model(model_int8)
model_int8 = None
print("\n✓  INT8 benchmark complete — model released")

In [ ]:
# — Cell 7: INT4 Quantization (NF4) ——————————————————
# NF4 = Normal Float 4-bit — the same quantization used in Chapter 6 QLoRA.
# 0.5 bytes per weight. Best compression; small quality penalty on
# knowledge-intensive tasks. Double quantization further reduces overhead.
# Expected: ~0.6 GB VRAM, comparable throughput to INT8 on T4.

print("Loading INT4 (NF4) model...")
vram_before = vram_used_gb()

bnb_int4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,   # quantize the quantization constants (extra ~0.1GB saving)
)
model_int4 = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_int4,
    device_map="auto",
)
model_int4.eval()

vram_after = vram_used_gb()
param_gb   = model_size_bytes(model_int4) / 1e9   # includes quant overhead

print(f"Benchmarking INT4  ({N_BENCH} runs)...")
stats = benchmark(model_int4, tokenizer, BENCH_PROMPT, N_BENCH, MAX_NEW)

results["int4"] = {
    "precision":     "INT4/NF4 (0.5 bytes/weight + double quant)",
    "vram_gb":       round(vram_after - vram_before, 2),
    "param_gb":      round(param_gb, 2),
    **stats,
}

print(f"  VRAM delta   : {results['int4']['vram_gb']:.2f} GB")
print(f"  Param size   : {results['int4']['param_gb']:.2f} GB")
print(f"  Latency mean : {stats['latency_mean']:.2f}s")
print(f"  Throughput   : {stats['tps_mean']:.1f} tok/s")
print(f"  Sample output: {stats['last_response'][:100]}...")

clear_model(model_int4)
model_int4 = None
print("\n✓  INT4 benchmark complete — model released")

In [ ]:
# — Cell 8: Precision Comparison Table ———————————————

print("=" * 72)
print("  QUANTIZATION BENCHMARK RESULTS — TinyLlama 1.1B")
print("=" * 72)
print(f"  {'Precision':<38} {'VRAM':>6}  {'Tok/s':>6}  {'Latency':>8}")
print("  " + "─" * 68)

fp16_tps  = results["fp16"]["tps_mean"]
fp16_vram = results["fp16"]["vram_gb"]

for key, r in results.items():
    tps_ratio  = r["tps_mean"]  / fp16_tps  if fp16_tps  > 0 else 1.0
    vram_ratio = r["vram_gb"]   / fp16_vram if fp16_vram > 0 else 1.0
    tps_tag    = f"({tps_ratio:+.0%} vs fp16)" if key != "fp16" else "(baseline)"
    vram_tag   = f"({vram_ratio:.0%} of fp16)" if key != "fp16" else "(baseline)"
    print(f"  {r['precision']:<38} {r['vram_gb']:>4.2f}GB  {r['tps_mean']:>6.1f}  {r['latency_mean']:>6.2f}s")
    print(f"  {'':38} {vram_tag:>6}  {tps_tag}")
    print()

print("=" * 72)
print()
print("QUALITY CHECK — same prompt, all three precisions:")
print("─" * 72)
print(f"  Prompt: {BENCH_PROMPT.split(chr(10))[1].strip()}")
print()
for key, r in results.items():
    label = key.upper().ljust(6)
    print(f"  [{label}]  {r['last_response'][:120]}")
    print()
print("  NOTE: All three precisions should give comparable answers on this")
print("  simple factual prompt. Quality diverges on longer, multi-step reasoning.")

In [ ]:
# — Cell 9: Model Size on Disk ———————————————————————
# VRAM usage during inference ≠ file size on disk.
# For deployment, disk size determines container image size, cold-start time,
# and storage cost in S3/ECR.

import subprocess, shutil, os

SAVE_PATHS = {
    "fp16":  "/content/size_test_fp16",
    "int8":  "/content/size_test_int8",
    "int4":  "/content/size_test_int4",
}

def disk_size_gb(path: str) -> float:
    """Total size of a directory in GB."""
    result = subprocess.run(['du', '-sb', path], capture_output=True, text=True)
    if result.returncode == 0:
        return int(result.stdout.split()[0]) / 1e9
    return 0.0

print("Saving each precision to disk to measure file size...")
print("(This takes ~1–2 minutes — saving safetensors for each)")
print()

configs = {
    "fp16": {"torch_dtype": torch.float16},
    "int8": {"quantization_config": BitsAndBytesConfig(load_in_8bit=True)},
    "int4": {"quantization_config": BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
    )},
}

disk_sizes = {}
for name, kwargs in configs.items():
    path = SAVE_PATHS[name]
    if not os.path.isdir(path):
        mdl = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto", **kwargs)
        mdl.save_pretrained(path, safe_serialization=True)
        tokenizer.save_pretrained(path)
        clear_model(mdl)
    disk_sizes[name] = disk_size_gb(path)
    print(f"  {name.upper():5s}  {disk_sizes[name]:.2f} GB  ({path})")

print()
print("DISK SIZE SUMMARY:")
print("─" * 45)
base = disk_sizes["fp16"]
for name, gb in disk_sizes.items():
    ratio = gb / base if base > 0 else 1.0
    bar   = "█" * int(ratio * 20)
    print(f"  {name.upper():5s}  {gb:.2f} GB  {bar}  ({ratio:.0%} of fp16)")

print()
print("✓  Smaller disk size = smaller Docker image = faster cold start")
print("   INT4 models fit in a free-tier S3 request vs INT8/fp16 needing larger transfer")

In [ ]:
# — Cell 10: Dynamic Quantization (CPU / Edge) ————————
# BitsAndBytesConfig requires CUDA. For CPU-only servers, IoT devices,
# or edge deployments, PyTorch's built-in dynamic quantization converts
# Linear layers to INT8 at inference time — no GPU required.
# Best for: small batches, latency-tolerant pipelines, edge inference.

print("Loading fp16 model for dynamic quantization demo...")
model_cpu = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float32,   # dynamic quant works on fp32 weights
    device_map="cpu",            # CPU only
)
model_cpu.eval()

size_before = model_size_bytes(model_cpu) / 1e9
print(f"  Model size before dynamic quant : {size_before:.2f} GB")

import torch.quantization as tq

model_dyn = tq.quantize_dynamic(
    model_cpu,
    {torch.nn.Linear},     # quantize all Linear layers
    dtype=torch.qint8,     # signed INT8
)

size_after = model_size_bytes(model_dyn) / 1e9
print(f"  Model size after  dynamic quant : {size_after:.2f} GB")
print(f"  Compression ratio               : {size_before / size_after:.1f}x")

# Quick CPU inference test
print("\nRunning CPU inference (slow — expected)...")
inputs = tokenizer(
    "### Instruction:\nWhat is Python?\n\n### Response:\n",
    return_tensors="pt", max_length=128, truncation=True
)
t0 = time.time()
with torch.no_grad():
    out = model_dyn.generate(
        **inputs, max_new_tokens=40, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
elapsed = time.time() - t0
text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(f"  Response  : {text[:100]}")
print(f"  CPU time  : {elapsed:.1f}s  ({(out.shape[1] - inputs['input_ids'].shape[1]) / elapsed:.1f} tok/s)")
print()
print("  USE WHEN : edge device, no NVIDIA GPU, latency > 1s acceptable")
print("  AVOID    : production GPU server — BitsAndBytesConfig int8/int4 is faster there")

clear_model(model_cpu)
clear_model(model_dyn)
print("\n✓  Dynamic quantization demo complete")

In [ ]:
# — Cell 11: Cost Calculator — $/1k Tokens ————————————
# GPU time is the dominant cost of LLM inference.
# Formula: cost_per_1k_tokens = (gpu_cost_per_hour / 3600) / (tokens_per_sec / 1000)

GPU_PRICES = {
    # Instance type         $/hr    Notes
    "T4 (Colab / g4dn)":   0.526,  # AWS g4dn.xlarge on-demand
    "A10G (g5.xlarge)":    1.006,  # AWS g5.xlarge on-demand
    "A100 40GB (p3.2xl)":  3.060,  # AWS p3.2xlarge (V100, close to A100)
    "A100 80GB (p4d)":     3.912,  # AWS p4d per-GPU fraction
    "GovCloud T4 equiv":   0.659,  # GovCloud ~25% premium vs commercial
    "GovCloud A10G equiv": 1.257,
}

def cost_per_1k(gpu_cost_per_hour: float, tokens_per_sec: float) -> float:
    """$/1k tokens = ($/hr ÷ 3600 s/hr) ÷ (tok/s ÷ 1000 tok/1k)"""
    if tokens_per_sec <= 0:
        return float("inf")
    cost_per_sec   = gpu_cost_per_hour / 3600
    tokens_per_sec_per_1k = tokens_per_sec / 1000
    return cost_per_sec / tokens_per_sec_per_1k

print("COST PER 1,000 TOKENS — MATRIX")
print("=" * 75)
print(f"  {'GPU Instance':<28}  {'$/hr':>6}  ", end="")
for key in results:
    print(f"  {key.upper():>8}", end="")
print()
print("  " + "─" * 70)

for gpu_label, gpu_price in GPU_PRICES.items():
    print(f"  {gpu_label:<28}  ${gpu_price:>5.3f}  ", end="")
    for key, r in results.items():
        cost = cost_per_1k(gpu_price, r["tps_mean"])
        print(f"  ${cost:>6.4f}", end="")
    print()

print("=" * 75)
print()
print("CONTEXT:")
print("  OpenAI GPT-4o input : $2.50 / 1M tokens  =  $0.0025 / 1k tokens")
print("  OpenAI GPT-4o output: $10.00 / 1M tokens =  $0.0100 / 1k tokens")
print()
print("  Self-hosted TinyLlama on T4 at the throughputs above:")
for key, r in results.items():
    c = cost_per_1k(0.526, r["tps_mean"])
    print(f"    {key.upper():5s}  ${c:.4f} / 1k tokens  ({c/0.01*100:.0f}% of GPT-4o output cost)")
print()
print("✓  Cost calculator complete")
print("   KEY INSIGHT: Self-hosted is cheaper at scale but has fixed GPU costs")
print("   Break-even point depends on your request volume and GPU utilisation rate")

In [ ]:
# — Cell 12: Precision Selection Guide ———————————————
# Nothing to execute — decision reference.

GUIDE = """
PRECISION SELECTION GUIDE
═════════════════════════════════════════════════════════════════════

  Precision      Bytes/weight  Use when
  ─────────────────────────────────────────────────────────────────
  fp32           4.0           Training only — too expensive for serving
  bf16           2.0           Training on A100/H100; better range than fp16
  fp16           2.0           ✓ Default production precision on T4/A10G
                               Fine for most inference; widest model support
  INT8           1.0           ✓ VRAM constrained but quality non-negotiable
                               Good for knowledge-intensive tasks (RAG, Q&A)
                               5–15% throughput penalty vs fp16
  INT4/NF4       0.5           ✓ Maximum compression; small quality penalty
  (BnB)                        Best for chat/instruction following (like Ch6)
                               Ch6 QLoRA training uses this precision
                               Not recommended for math / code generation
  Dynamic INT8   1.0 (CPU)     ✓ CPU-only edge/IoT; no NVIDIA GPU required
  (PyTorch)                    2–5× slower than GPU INT8

  ─────────────────────────────────────────────────────────────────

  DECISION TREE:

  GPU available?
    No  → Dynamic quantization (PyTorch) on CPU
    Yes → Do you need the merged LoRA adapter (Ch10)?
            Yes → Load as fp16 with merge_and_unload() — adapters can't merge from INT8/INT4
            No  →
              VRAM tight (≤8 GB GPU)?
                Yes → INT4/NF4 (bitsandbytes)
                No  →
                  Quality is top priority (RAG, documents)?  → INT8
                  Speed / cost is top priority?              → INT4/NF4

  ─────────────────────────────────────────────────────────────────

  GOVCLOUD NOTES:
  • AWS GovCloud GPU instances mirror commercial: g4dn (T4), g5 (A10G), p4d (A100)
  • GovCloud on-demand pricing is ~20–25% above us-east-1 commercial
  • Spot/interruptible instances save 60–80% — acceptable for batch workloads
  • For the DocuMind Gov enclave: INT4 fits entirely within a g4dn.xlarge (16 GB VRAM)
    leaving 15+ GB headroom for the RAG FAISS index and FastAPI overhead
  • Recommendation for Phase II: g4dn.xlarge spot instance + INT4 model
    Estimated cost: ~$0.19/hr spot × 8 hr/day = ~$1.50/day = ~$45/month

  ─────────────────────────────────────────────────────────────────

  WHAT QUANTIZATION CANNOT FIX:
  • A bad model — garbage in, garbage out at any precision
  • Missing knowledge — use RAG (Ch8) to inject facts, not more bits
  • Context length — quantization doesn't extend the model's window
  • Alignment — DPO (Ch7) / guardrails (Ch12) handle safety, not bit-width
"""

print(GUIDE)
print("✓  Chapter 13 complete")

## Chapter 13 Complete ✓

**What happened:**
- Loaded TinyLlama in fp16, INT8, and INT4 precision; measured VRAM usage, tokens/sec, and latency for each
- Saved each precision to disk and measured file size (critical for container image sizing)
- Applied PyTorch dynamic quantization for CPU/edge deployment without NVIDIA GPU
- Computed cost per 1,000 tokens across six GPU instance types (T4 → A100, commercial + GovCloud)
- Printed a precision selection decision tree covering every deployment scenario

**The one-number summary:**
> INT4/NF4 uses **4× less VRAM** than fp16, runs at **comparable throughput** on T4,
> and costs **4× less per token** — with a small but measurable quality penalty on
> complex reasoning tasks. For chat and instruction following (the chat / instruction following use case),
> the penalty is negligible.

**Cost optimisation beyond quantization (not demoed here):**
- **Continuous batching** (vLLM): merge concurrent requests → 10–50× throughput gain at the same per-GPU cost
- **Speculative decoding**: small draft model proposes tokens; large model verifies in parallel → 2–3× speedup
- **KV cache quantization** (INT8 KV): reduces memory bandwidth for long contexts
- **Spot/preemptible instances**: 60–80% cost reduction for batch and async workloads

**GovCloud recommendation for Phase II deployment:**
`g4dn.xlarge` spot instance + INT4/NF4 model + vLLM continuous batching
Estimated: ~$0.19/hr spot × utilisation factor → < $50/month for dev/staging workload

**Next: Chapter 14 — Multi-Modal & Agentic AI**